# Speech Translation

In [ ]:
!pip install googletrans==4.0.0-rc1

In [ ]:
# implementing for text basis....

# Import the necessary module
from googletrans import Translator

# Initialize the translator
translator = Translator()

# Supported languages dictionary
languages = {
    'en': 'English',
    'hi': 'Hindi',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'zh-cn': 'Chinese (Simplified)',
    'ar': 'Arabic',
    'ja': 'Japanese'
}

# Default source language (user's spoken language)
default_src_lang = 'hi'  # Adapt this based on the user's language preference

# Function to translate text
def translate_text():
    # Ask user for input text
    text = input(f"Enter the text you want to translate from {languages[default_src_lang]}: ")

    # Display the supported languages
    print("Supported languages:")
    for code, lang in languages.items():
        print(f"{code}: {lang}")

    # Ask user for target language, default to English
    dest_lang = input("Enter the target language code (default is 'en' for English): ").strip().lower() or 'en'

    if dest_lang not in languages:
        print("Invalid target language code.")
        return

    # Perform translation
    translated = translator.translate(text, src=default_src_lang, dest=dest_lang).text
    print(f"Translation from {languages[default_src_lang]} to {languages[dest_lang]}: {translated}")

# Run the function
translate_text()


In [ ]:
!pip install gtts googletrans==4.0.0-rc1 speechrecognition pydub

In [ ]:
!pip install sounddevice googletrans==4.0.0-rc1 gtts speechrecognition pydub

In [ ]:
!pip install sounddevice wavio SpeechRecognition

In [ ]:
# implementing for voice basis....

import sounddevice as sd
import wavio
import speech_recognition as sr
from googletrans import Translator
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
import os

# Initialize the translator
translator = Translator()

# Supported languages dictionary
languages = {
    'en': 'English',
    'hi': 'Hindi',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'zh-cn': 'Chinese (Simplified)',
    'ar': 'Arabic',
    'ja': 'Japanese'
}

# Default source language (user's spoken language)
default_src_lang = 'hi'  # Adapt this based on the user's language preference
default_dest_lang = 'en'  # Default translation target language is English

# Function to record audio
def record_audio(filename, duration=5, fs=16000):
    print(f"Recording for {duration} seconds...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()  # Wait until the recording is finished
    wavio.write(filename, recording, fs, sampwidth=2)
    print("Recording complete.")

# Function to recognize speech and translate
def recognize_and_translate():
    audio_filename = "./test-recording/input/input.wav"
    
    # Record audio from the microphone
    record_audio(audio_filename)
    
    recognizer = sr.Recognizer()
    
    with sr.AudioFile(audio_filename) as source:
        audio = recognizer.record(source)
        
        try:
            # Recognize speech using Google's speech recognition
            text = recognizer.recognize_google(audio, language=default_src_lang)
            print(f"Recognized text: {text}")
            
            # Translate the recognized text
            translated_text = translator.translate(text, src=default_src_lang, dest=default_dest_lang).text
            print(f"Translation to {languages[default_dest_lang]}: {translated_text}")
            
            # Convert translated text to speech
            tts = gTTS(translated_text, lang=default_dest_lang)
            tts.save("translated.mp3")
            
            # Play the translated speech
            translated_audio = AudioSegment.from_mp3("./test-recording/translated/translated.mp3")
            play(translated_audio)
            os.remove("translated.mp3")
            
        except sr.UnknownValueError:
            print("Could not understand the audio.")
        except sr.RequestError as e:
            print(f"Error with the speech recognition service; {e}")

# Run the function
recognize_and_translate()


In [ ]:
pip install langdetect

In [ ]:
pip install sounddevice wavio SpeechRecognition googletrans==4.0.0-rc1 langdetect gtts pydub

In [ ]:
import sounddevice as sd
import wavio
import speech_recognition as sr
from googletrans import Translator, LANGUAGES
from langdetect import detect
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
import os

# Initialize the translator
translator = Translator()

# Function to record audio
def record_audio(filename, duration=5, fs=16000):
    print(f"Recording for {duration} seconds...")
    recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()  # Wait until the recording is finished
    wavio.write(filename, recording, fs, sampwidth=2)
    print("Recording complete.")

# Function to detect the language of text using langdetect
def detect_language(text):
    try:
        detected_lang = detect(text)
        detected_language_name = LANGUAGES.get(detected_lang, 'Unknown')
        print(f"Detected language: {detected_language_name}")
        return detected_lang
    except:
        print("Language detection failed.")
        return None

# Function to recognize speech and translate based on detected language
def recognize_and_translate(user_label, target_lang):
    audio_filename = f"./input_{user_label}.wav"
    
    # Record audio from the microphone
    record_audio(audio_filename)
    
    recognizer = sr.Recognizer()
    
    with sr.AudioFile(audio_filename) as source:
        audio = recognizer.record(source)
        
        try:
            # Recognize speech using Google's speech recognition
            text = recognizer.recognize_google(audio)
            print(f"{user_label} said: {text}")
            
            # Detect the language of the recognized text
            src_lang = detect_language(text)
            if src_lang is None:
                print(f"{user_label}: Could not detect language. Skipping translation.")
                return None, None
            
            # Translate the recognized text to the target language
            translated_text = translator.translate(text, src=src_lang, dest=target_lang).text
            print(f"Translation to {LANGUAGES.get(target_lang, 'Unknown')}: {translated_text}")
            
            # Convert translated text to speech
            tts = gTTS(translated_text, lang=target_lang)
            translated_audio_file = f"./translated_{user_label}.mp3"
            tts.save(translated_audio_file)
            
            # Play the translated speech
            translated_audio = AudioSegment.from_mp3(translated_audio_file)
            play(translated_audio)
            os.remove(translated_audio_file)
            
            return src_lang, target_lang
        
        except sr.UnknownValueError:
            print(f"{user_label}: Could not understand the audio.")
        except sr.RequestError as e:
            print(f"{user_label}: Error with the speech recognition service; {e}")
        
        return None, None

# Function to manage conversation between two users
def conversation():
    # Placeholder for the detected languages of both users
    user_1_lang = None
    user_2_lang = None
    
    while True:
        print("\nListening to User 1...")
        if user_2_lang:
            # Translate User 1's input to User 2's language
            user_1_lang, _ = recognize_and_translate("User 1", user_2_lang)
        else:
            # Identify and store User 1's language in the first round
            user_1_lang, _ = recognize_and_translate("User 1", "en")  # Default to English if no language is detected

        print("\nListening to User 2...")
        if user_1_lang:
            # Translate User 2's input to User 1's language
            user_2_lang, _ = recognize_and_translate("User 2", user_1_lang)
        else:
            # Identify and store User 2's language in the first round
            user_2_lang, _ = recognize_and_translate("User 2", "en")  # Default to English if no language is detected
        
        # Option to continue or end the conversation
        cont = input("\nDo you want to continue the conversation? (yes/no): ").strip().lower()
        if cont != 'yes':
            break

# Run the conversation
conversation()
